# Sesión 06 — Notebook conceptual

Este notebook acompaña los momentos NO-live-coding de la Sesión 06. La sesión es **flipped classroom estricto**: los 5 articles del módulo ya están leídos. Aquí solo se aplican.

**Cronometría sobre la que vamos a operar:**

| Bloque | Tema | Duración |
|--------|------|----------|
| 0      | Apertura y anclaje en el ejercicio pre-sesión | 0:00 – 0:10 (10 min) |
| 1      | Resolución de errores | 0:10 – 0:25 (15 min) |
| 2.1    | Decisión arquitectónica CAG/RAG/híbrido | 0:25 – 0:40 (15 min) |
| 2.2    | Catálogo de fuentes | 0:40 – 1:00 (20 min) — live coding en `app/` |
| 2.3    | Subsistema de ingesta + `Document` | 1:00 – 1:20 (20 min) — live coding en `app/` |
| 2.4    | Limpieza + Pandera | 1:20 – 1:35 (15 min) — live coding en `app/` |
| 2.5    | Pseudonimización con Presidio | 1:35 – 1:55 (20 min) — live coding en `app/` |
| 3      | Cierre + bridge a S07 | 1:55 – 2:00 (5 min) |

Las celdas conceptuales viven aquí; el código de servicio vive en `app/ingestion/...`. Cuando una celda dice "abrir `app/...`", el live coding se hace fuera del notebook.

## Bloque 0 — Apertura y anclaje (0:00 – 0:10)

Recogida en chat:

> ¿Cuál de las cuatro restricciones del CAG se rompió primero en tu ejercicio pre-sesión, y con qué número?

Tabular en vivo 4–5 respuestas. La tabla siguiente queda como acta del directo.

In [ ]:
# Cabecera para anotar en vivo durante la apertura. NO ejecutar antes — se rellena con los hallazgos del chat.
import pandas as pd

findings = pd.DataFrame(
    columns=[
        "alumno",
        "restriccion_rota_primero",  # context_window | cost | latency | lost_in_the_middle
        "numero_concreto",            # tokens, ms, $/turn, drift event description
        "escenario",                  # growing | pivot | contradiction
        "notas",
    ]
)
findings

**Pregunta trampa para el chat:** ¿alguno detectó *lost in the middle*?

Si nadie lo trae, introducirlo en voz como modo de fallo sutil que la prueba directa no detecta — el modelo "responde" pero ignora información del centro del contexto. Conecta con el motivo del módulo entero: la solución no es "más contexto", es **datos curados**.

**Cierre del bloque (literal):**
> "Estos números no son anécdotas, son la justificación arquitectónica del módulo entero. Hoy convertimos ese diagnóstico en cinco piezas: catálogo, ingesta, Document canónico, validación, pseudonimización. Vamos."

## Sub-bloque 2.1 — Decisión arquitectónica (0:25 – 0:40)

Article 1 del módulo. Dos piezas:

1. `CAGViability` — dataclass de viabilidad sobre las 4 restricciones (context_window, cost, latency, lost_in_the_middle).
2. `recommend_architecture(corpus, model)` — función que evalúa los 4 ejes (volumen, frecuencia, trazabilidad, sensibilidad) y devuelve `"CAG" | "Hybrid" | "RAG"`.

**Defensas orales para este sub-bloque:**
- Los cuatro ejes son **AND** y no **OR** — un solo eje en rojo basta para descartar CAG puro.
- Fine-tuning **no** aparece en el árbol; es una capa ortogonal que no resuelve trazabilidad.
- Tres opciones (CAG/híbrido/RAG), no dos — el híbrido es la opción que más empresas adoptan al final.

In [ ]:
# LIVE: el instructor escribe esta celda. Article 1, dataclass de viabilidad CAG.
from dataclasses import dataclass

@dataclass
class CAGViability:
    context_window_ok: bool      # ¿cabe el corpus completo en el context window?
    cost_ok: bool                # ¿el coste por turno con prefix caching es asumible?
    latency_ok: bool             # ¿la latencia con el corpus completo se queda bajo SLO?
    lost_in_the_middle_ok: bool  # ¿los benchmarks muestran que el modelo USA el centro?

    @property
    def viable(self) -> bool:
        return all([self.context_window_ok, self.cost_ok, self.latency_ok, self.lost_in_the_middle_ok])

    def failing_constraints(self) -> list[str]:
        return [name for name, ok in [
            ("context_window", self.context_window_ok),
            ("cost", self.cost_ok),
            ("latency", self.latency_ok),
            ("lost_in_the_middle", self.lost_in_the_middle_ok),
        ] if not ok]

In [ ]:
# LIVE: CorpusProfile y ModelProfile del Proyecto 2 con números defendibles.
@dataclass
class CorpusProfile:
    name: str
    estimated_tokens: int            # volúmen total estimado
    refresh_frequency_days: float    # cada cuánto cambian los datos
    traceability_required: bool      # ¿hay que poder citar la fuente?
    access_control_required: bool    # ¿hay que filtrar por permisos por usuario?

@dataclass
class ModelProfile:
    name: str
    context_window: int       # tokens
    cost_per_1k_input: float  # USD
    prefix_caching: bool      # ¿soporta caché de prefijo?

proyecto_2 = CorpusProfile(
    name="Proyecto 2",
    estimated_tokens=250_000,         # presupuestos + transcripciones + tarifas + adendas
    refresh_frequency_days=7,         # cierre comercial semanal
    traceability_required=True,       # legal exige citar la fuente
    access_control_required=True,     # info confidencial cliente
)

modelo_actual = ModelProfile(
    name="gpt-4o-mini",
    context_window=128_000,
    cost_per_1k_input=0.00015,
    prefix_caching=True,
)

proyecto_2, modelo_actual

In [ ]:
# LIVE: función de recomendación arquitectónica sobre los 4 ejes.
from typing import Literal

def recommend_architecture(
    corpus: CorpusProfile, model: ModelProfile
) -> Literal["CAG", "Hybrid", "RAG"]:
    # Eje 1: VOLUMEN.
    fits_in_window = corpus.estimated_tokens <= model.context_window * 0.7

    # Eje 2: FRECUENCIA. Cambios más frecuentes que semanales rompen prefix caching.
    cache_friendly = corpus.refresh_frequency_days >= 7

    # Eje 3: TRAZABILIDAD. Sin retriever no hay metadata de fuente que citar.
    traceability_doable = not corpus.traceability_required

    # Eje 4: CONTROL DE ACCESO. CAG no puede filtrar por usuario antes del prompt.
    access_control_doable = not corpus.access_control_required

    viable_for_cag = all([
        fits_in_window,
        cache_friendly,
        traceability_doable,
        access_control_doable,
    ])

    if viable_for_cag:
        return "CAG"
    # Si solo falla la trazabilidad/control de acceso, el modo híbrido (CAG
    # para glosarios estables + RAG para datos sensibles/citables) sigue siendo
    # razonable.
    if fits_in_window and cache_friendly:
        return "Hybrid"
    return "RAG"

recommend_architecture(proyecto_2, modelo_actual)

**Resultado esperado:** `"RAG"`.

**Discusión oral:**
- 250K tokens no son demasiado para 128K de window, pero recursos en el centro **se pierden** ("lost in the middle").
- Volveremos a usar CAG para una pieza residual: **glosario interno, plantillas de propuesta y tarifas estables**. Eso es lo que llamamos **arquitectura híbrida residual**: lo estable se queda CAG, lo voluminoso/sensible va a RAG.
- En las próximas sesiones construiremos esa pieza RAG. Hoy preparamos el corpus para que sea **vectorizable** — la pieza siguiente del puzle.

## Sub-bloque 2.5 — Las 3 tarjetas de filtración semántica (1:35 – 1:40 conceptual)

Article 5. Antes de saltar a Presidio, fijar el porqué con tres ejemplos del Proyecto 2.

**Tarjeta 1 — Filtración directa**
> Pregunta: *¿Qué dijo Laura Fernández sobre el despliegue?* → el LLM accede al nombre `Laura Fernández` literal en el corpus.
>
> Defensa: este es el modo OBVIO. Pseudonimizar el nombre antes del embedding rompe la búsqueda directa por valor.

**Tarjeta 2 — Filtración por agregación**
> Pregunta: *¿Qué cliente firma más presupuestos en Madrid?* → el LLM combina varios campos (nombre, ciudad, frecuencia) para inferir un perfil.
>
> Defensa: aquí el control de acceso por documento ya no protege. Si el usuario tiene acceso a 10 documentos, el LLM puede agregar 10 piezas y revelar algo que ninguna pieza por separado revelaba.

**Tarjeta 3 — Filtración por inferencia**
> Pregunta: *¿Qué cliente parece estar a punto de cancelar?* → el LLM no necesita ningún campo explícito; usa **señales** (tono, cadencia, términos como "reevaluando", "competencia") para inferir.
>
> Defensa: este es el modo MÁS PELIGROSO. Ni la pseudonimización cubre esto: la señal está en el lenguaje, no en los identificadores. Mitigarlo requiere ´grounding´ explícito ("responde solo lo que está en el documento"), classifier de PII contextual y revisión humana antes de exponer el output.

**Provocación al chat:** *¿Cuál es la más peligrosa de las tres y por qué?*

**Cierre conceptual:** el control de acceso del backend Rails **no** protege contra estos modos. La defensa tiene que ocurrir **antes del embedding**, dentro del servicio IA — que es lo que vamos a construir ahora.

## Bloque 3 — Cierre del módulo (1:55 – 2:00)

**Recap de las cinco piezas construidas hoy:**

- [ ] **Catálogo versionado** — `data/catalog/catalog.yaml` con 3 fuentes auditadas (1 include, 1 review, 1 exclude) y loader Pydantic.
- [ ] **Subsistema de ingesta** — `app/ingestion/{loaders,parsers}` con `Parser` Protocol y 2 parsers reales (JSON y TXT).
- [ ] **`Document` canónico** — `app/ingestion/documents/models.py` con `DocumentMetadata` propagado desde el catálogo.
- [ ] **Validación con Pandera** — `app/ingestion/cleaning/` con limpieza + `BudgetRecord` + `validate_with_policy` (reparar/cuarentena/descartar).
- [ ] **Pseudonimización con mapping table** — `app/ingestion/pii/` con Presidio en español, recognizers custom y mapping table persistente en Postgres.

**Estado del corpus al final del módulo:** defendible ante legal, comercial, técnico y regulatorio. Cada decisión versionada, cada exclusión con motivo, cada dato sensible con mapping reversible.

**Bridge a Sesión 07:** *chunking, embeddings, espacio vectorial*. El trabajo de hoy es el cimiento.

**Entregas:**
- Repo de referencia completo → `guides/session-06-reference/`.
- Pre-flight script para validar el setup → `scripts/preflight_s06.py`.
- Cliente Rails ilustrativo → `estimator-web/app/services/estimator_ai/ingestion_client.rb`.